# **LAB511**: Create advanced Postgres-powered agentic apps with Azure HorizonDB

## Notebook 1 - Data Setup

### Part 2.1: Introduction

**Welcome to the LAB511 Data Setup Notebook!**

In this notebook you will stand up the sample dataset that powers the agentic app you'll build in Notebook 2. Working through it, you will:

- Connect to your **Azure HorizonDB** instance and enable the extensions that turn Postgres into a multi-model AI platform - `vector`, `pg_diskann`, `pg_fts`, `age`, and `azure_ai`.
- Load a corpus of U.S. court cases from `cases.csv` into a clean relational schema.
- Generate 1536-dimension embeddings for every opinion with **Azure OpenAI** and store them alongside the source rows.
- Build a citation **property graph** with Apache AGE, plus full-text (`pg_fts`) and ANN (`pg_diskann`) indexes - the three retrieval modes your agent will combine for hybrid search.

By the end you will have one Postgres database that serves relational, vector, full-text, and graph queries - no separate stores, no ETL fan-out.

**Database Diagram**

**Graph Diagram**


### Part 2.2: Setup the Data Loading Python imports

##### 🧠 *Technical Background Notes*

These imports set up the three pillars used throughout the rest of the lab:

1. **Database access** - `psycopg` is the modern Postgres driver. Every cell below opens a connection with `psycopg.connect(**DB_CONFIG)` to run SQL, `COPY` the CSV, write `vector(1536)` embeddings, and build the AGE graph.
2. **Azure OpenAI access** - `AzureOpenAI` (from `openai`) is the client used later to generate embeddings for each case opinion and populate the `opinions_vector` column.
3. **Configuration** - `load_dotenv` + `os` pull Azure OpenAI and Postgres credentials from your `.env` into `os.environ` so nothing is hard-coded.

Two small helpers round it out: `nest_asyncio` lets `async` code run inside Jupyter's already-running event loop (essential in the next notebook when we call agents), and `time` is used to report progress and duration of the embedding batch job.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. This will run the code and show the output below.  Since these are just imports, there is nothing to show at the end other than a check mark showing success.

    > **Note:** The first time this code block is ran, it may take around 10-15 seconds.

In [ ]:
# Allow nested event loops (needed for async code inside Jupyter)
import nest_asyncio
nest_asyncio.apply()

# Standard library
import os
import time

# Database driver
import psycopg

# Azure OpenAI client
from openai import AzureOpenAI

# Environment variable loading
from dotenv import load_dotenv

### Part 2.3: Load environment variables and build the DB config

##### 🧠 *Technical Background Notes*

`load_dotenv(override=True)` reads your local `.env` and pushes every key into `os.environ`, overriding any stale shell values - important when you re-provision and your endpoint or password changes.

We then pull two logical groups out of the environment:

- **Azure OpenAI settings** (`AZURE_OPENAI_*`, `AZURE_EMBED_DEPLOYMENT`, `AZURE_API_VERSION`) - feed the `AzureOpenAI` client later when we generate embeddings.
- **`DB_CONFIG` dict** - unpacked via `**DB_CONFIG` into every `psycopg.connect()` call in this notebook. `sslmode` defaults to `require` because HorizonDB only accepts TLS connections.

The `print` calls are a quick sanity check that nothing is `None`; remove or comment them out before sharing a notebook - they will echo your key to the cell output.


##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. Confirm the three printed lines show your chat deployment, embedding deployment, and API version - no `None` values.

In [ ]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT   = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
AZURE_OPENAI_KEY        = os.environ["AZURE_OPENAI_KEY"]
AZURE_EMBED_DEPLOYMENT  = os.environ["AZURE_EMBED_DEPLOYMENT"]
AZURE_API_VERSION       = os.environ["AZURE_API_VERSION"]

DB_CONFIG = {
    "host":     os.environ["AZURE_PG_HOST"],
    "dbname":   os.environ["AZURE_PG_NAME"],
    "user":     os.environ["AZURE_PG_USER"],
    "password": os.environ["AZURE_PG_PASSWORD"],
    "port":     os.environ["AZURE_PG_PORT"],
    "sslmode":  os.environ.get("AZURE_PG_SSLMODE", "require"),
}

print(AZURE_OPENAI_DEPLOYMENT)
print(AZURE_EMBED_DEPLOYMENT)
print(AZURE_API_VERSION)


### Part 2.4: Install the Postgres extensions used by the lab

##### 🧠 *Technical Background Notes*

Every advanced capability used in this lab is unlocked by a Postgres extension. `CREATE EXTENSION IF NOT EXISTS ... CASCADE` is idempotent, so the cell is safe to re-run.

- **`azure_ai`** - invoke Azure OpenAI from inside SQL (used later for in-database generation/embedding).
- **`vector`** - adds the `vector` data type and similarity operators (`<->`, `<#>`, `<=>`).
- **`age`** - Apache AGE, layers a Cypher property graph on top of Postgres; powers the `case_graph` we build below.
- **`pg_diskann`** - Microsoft's DiskANN ANN index for high-recall, low-latency vector search at scale.
- **`pg_fts`** - modern full-text search index (`text_fts_ops`) used by the hybrid search query later.

`autocommit = True` is required because `CREATE EXTENSION` cannot run inside a user transaction block.


##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. Watch for five `... extension ready.` lines followed by `Done. Extensions Created.`

In [ ]:
# Create Extensions Needed for Lab
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    
    cur.execute("CREATE EXTENSION IF NOT EXISTS azure_ai CASCADE;")
    print("AZURE_AI extension ready.")

    cur.execute("CREATE EXTENSION IF NOT EXISTS vector CASCADE;")
    print("VECTOR extension ready.")

    cur.execute("CREATE EXTENSION IF NOT EXISTS age CASCADE;")
    print("AGE extension ready.")

    cur.execute("CREATE EXTENSION IF NOT EXISTS pg_diskann CASCADE;")
    print("DISKANN extension ready.")

    cur.execute("CREATE EXTENSION IF NOT EXISTS pg_fts CASCADE;")
    print("PG_FTS extension ready.")

    #cur.execute("CREATE EXTENSION IF NOT EXISTS pg_durable CASCADE;")
    #print("PG_DURABLE extension ready.")    

conn.close()
print("Done. Extensions Created.")

### Part 2.5: Register models with the `azure_ai` model registry

##### 🧠 *Technical Background Notes*

The `azure_ai` extension lets you call Azure OpenAI directly from SQL (e.g. `azure_openai.create_embeddings(...)`, `azure_ai.generate(...)`). Before it can do that it needs to know **which** Azure OpenAI deployments to call and **how** to authenticate.

Instead of the legacy `azure_ai.set_setting(...)` approach (single global endpoint + key), we register each deployment in the **model registry** with `model_registry.model_add(...)`. Each registered model gets a short **alias** that callers pass to the AI functions, e.g. `azure_openai.create_embeddings('lab-embedding', ...)` or `azure_ai.generate(..., 'lab-chat')`.

For each model we provide:

- a unique **alias** used from SQL,
- the Azure OpenAI **endpoint URL** (`https://<resource>.openai.azure.com/`),
- the **deployment name** and **model name**,
- the **API version**,
- the **auth type** (`subscription-key`) and the **endpoint key**.

We register two models here - one chat-completion deployment (`lab-chat`) and one embedding deployment (`lab-embedding`) - reusing the values already loaded from `.env` in Part 2.3. Values are bound as parameters (`%s`) so the key never appears in the SQL text.

To keep the cell safely re-runnable, we first call `model_registry.model_remove(alias)` for each alias (wrapped in `try/except` because removing a model that doesn't exist raises an error on the first run). After registering both models, we read them back via `model_registry.model_list_all()` to confirm the rows landed correctly.

`autocommit = True` is on so the registrations are persisted immediately and visible to every later connection in this notebook.


##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. The verification rows printed from `model_list_all()` should show your two aliases with the correct endpoint and deployment names.

1. Confirm both `Registered chat model alias: lab-chat` and `Registered embedding model alias: lab-embedding` print.

In [ ]:
# Register Azure OpenAI deployments in the azure_ai model registry
CHAT_MODEL_ALIAS  = "lab-chat"
EMBED_MODEL_ALIAS = "lab-embedding"

conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    # Remove any prior registrations so this cell is safely re-runnable
    for alias in (CHAT_MODEL_ALIAS, EMBED_MODEL_ALIAS):
        try:
            cur.execute("SELECT model_registry.model_remove(%s);", (alias,))
        except Exception:
            pass

    # Register the chat-completion deployment
    cur.execute(
        """
        SELECT model_registry.model_add(
            %s,                 -- model alias
            %s,                 -- endpoint URL
            %s,                 -- deployment name
            %s,                 -- model name
            %s,                 -- API version
            'subscription-key', -- auth type
            %s                  -- endpoint key
        );
        """,
        (
            CHAT_MODEL_ALIAS,
            AZURE_OPENAI_ENDPOINT,
            AZURE_OPENAI_DEPLOYMENT,
            AZURE_OPENAI_DEPLOYMENT,
            AZURE_API_VERSION,
            AZURE_OPENAI_KEY,
        ),
    )
    print(f"Registered chat model alias:      {CHAT_MODEL_ALIAS}")

    # Register the embedding deployment
    cur.execute(
        """
        SELECT model_registry.model_add(
            %s, %s, %s, %s, %s, 'subscription-key', %s
        );
        """,
        (
            EMBED_MODEL_ALIAS,
            AZURE_OPENAI_ENDPOINT,
            AZURE_EMBED_DEPLOYMENT,
            AZURE_EMBED_DEPLOYMENT,
            AZURE_API_VERSION,
            AZURE_OPENAI_KEY,
        ),
    )
    print(f"Registered embedding model alias: {EMBED_MODEL_ALIAS}")

    # Verify the registrations
    cur.execute(
        """
        SELECT alias, endpoint, deployment_name, model_name, api_version, auth_type        
        FROM model_registry.model_list_all()        
        ORDER BY alias;
        """        
    )
    for row in cur.fetchall():
        print(row)

conn.close()
print("Done. azure_ai models registered from environment variables.")


### Part 2.6: Create the schema and load `cases.csv`

##### 🧠 *Technical Background Notes*

The raw dataset is one big JSON document per row. Rather than write a parser in Python we let Postgres do the work:

1. **Two tables, two jobs** - `temp_cases(data jsonb)` is a staging table that holds the raw JSON; `cases` is the clean, typed table our app queries.
2. **Streaming `COPY`** - `cur.copy(...)` + an 8 KB read loop streams the CSV into `temp_cases` without buffering the whole file in memory. `COPY` is dramatically faster than row-by-row `INSERT`.
3. **JSON → columns in SQL** - the `INSERT ... SELECT` uses `jsonb` path operators (`#>>`, `jsonb_path_query`) to extract `id`, `name_abbreviation`, `decision_date`, `court.name`, and concatenate every opinion text into a single column. Doing this server-side keeps the dataset on disk and avoids network round-trips.

`temp_cases` is intentionally kept around - the graph cell later reuses its raw `cites_to` JSON to build citation edges.


##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. The cell prints the resolved CSV path and finishes with `Done. <N> cases loaded into the cases table.`

    > **Note:** Streaming and parsing the CSV typically takes 30-60 seconds.

In [ ]:
# Initialize dataset: create tables and load cases from CSV
csv_path = os.path.join(os.path.dirname(os.getcwd()), "Dataset", "cases.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(os.getcwd(), "Dataset", "cases.csv")
print(f"CSV path: {csv_path}")

conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    # Drop existing tables
    cur.execute("DROP TABLE IF EXISTS cases;")
    cur.execute("DROP TABLE IF EXISTS temp_cases;")

    # Create the cases table
    cur.execute("""
        CREATE TABLE cases(
            id SERIAL PRIMARY KEY,
            name TEXT,
            decision_date DATE,
            court_level TEXT,
            opinion TEXT
        );
    """)

    # Create temp staging table and load CSV
    cur.execute("CREATE TABLE temp_cases(data jsonb);")

    with open(csv_path, "r", encoding="utf-8") as f:
        with cur.copy("COPY temp_cases (data) FROM STDIN WITH (FORMAT csv, HEADER true)") as copy:
            while chunk := f.read(8192):
                copy.write(chunk)

    print("CSV loaded into temp_cases.")

    # Insert parsed data into cases table
    cur.execute("""
        INSERT INTO cases
        SELECT
            (data#>>'{id}')::int AS id,
            (data#>>'{name_abbreviation}')::text AS name,
            (data#>>'{decision_date}')::date AS decision_date,
            (data#>>'{court,name}')::text AS court_level,
            array_to_string(
                ARRAY(SELECT jsonb_path_query(data, '$.casebody.opinions[*].text')),
                ', '
            ) AS opinion
        FROM temp_cases;
    """)

    cur.execute("SELECT count(*) FROM cases;")
    count = cur.fetchone()[0]
    print(f"Done. {count} cases loaded into the cases table.")

conn.close()

### Part 2.7: Quick sanity check - peek at the loaded data

##### 🧠 *Technical Background Notes*

A five-row smoke test. If this returns nothing, the `COPY` above silently failed (usually a CSV path or encoding issue). Note that this connection is **not** `autocommit=True` - pure `SELECT` work doesn't need it, and the implicit read transaction is closed when we call `conn.close()`.


##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. You should see five rows printed - one per case. If nothing prints, the load in the previous cell didn't actually populate the `cases` table.

In [ ]:
# Test query: select top 5 rows from cases
conn = psycopg.connect(**DB_CONFIG)
with conn.cursor() as cur:
    cur.execute("SELECT * FROM cases LIMIT 5;")
    rows = cur.fetchall()
    for row in rows:
        print(row)
conn.close()

### Part 2.8: Add the `opinions_vector` column

##### 🧠 *Technical Background Notes*

`vector(1536)` matches the output dimensionality of `text-embedding-3-small`. The dimension is **fixed at column creation** - switching to a different embedding model later means dropping and re-creating this column.

`ADD COLUMN IF NOT EXISTS` makes this cell safe to re-run. The column is nullable so we can populate it in batches in the next cell rather than blocking the whole load.


##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. A single `opinions_vector column added.` line confirms the schema change succeeded.

In [ ]:
# Add the vector column to the cases table
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute("ALTER TABLE cases ADD COLUMN IF NOT EXISTS opinions_vector vector(1536);")    
conn.close()

print("opinions_vector column added.")

### Part 2.9: Generate and store the opinion embeddings

##### 🧠 *Technical Background Notes*

This is the longest-running cell in the notebook - it calls Azure OpenAI for every case. A few details worth knowing:

1. **Batching** - `BATCH_SIZE = 20` rows per embeddings request. The Azure OpenAI endpoint accepts an array of inputs and returns an array of vectors, which is ~20× faster than one call per row and stays well under the per-request token cap.
2. **Resumable** - the loop only selects rows `WHERE opinions_vector IS NULL`, so if the cell is interrupted you can simply re-run it and it will pick up where it left off. Set `CLEAR_VECTORS_BEFORE_RUN = False` to take advantage of that.
3. **Truncation** - opinion text is sliced to 8000 characters before sending. This keeps each input under the embedding model's token limit; for production you'd chunk and average instead of truncate.
4. **Vector literal** - pgvector accepts a string of the form `'[v1,v2,...]'::vector`. We build it with `",".join(...)` and cast on the server.

The timing prints exist so you can budget: in the lab environment expect roughly a minute per few hundred rows depending on opinion length.


##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. Watch the `Progress: X / Y (Z%)` lines print after each batch of 20 rows. The cell ends with a total duration summary.

    > **Note:** This is the longest-running cell in the notebook - expect several minutes for the full dataset.

In [ ]:
BATCH_SIZE = 20
CLEAR_VECTORS_BEFORE_RUN = True  # Set to False to skip clearing existing vectors

# Azure OpenAI client for embeddings
embed_client = AzureOpenAI(
    api_key=AZURE_OPENAI_KEY,
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)

# Connect and fetch rows that still need embeddings
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

# Optionally clear all existing vectors
if CLEAR_VECTORS_BEFORE_RUN:
    with conn.cursor() as cur:
        cur.execute("UPDATE cases SET opinions_vector = NULL")
    print("Cleared all existing vectors.")

start_time = time.time()
print(f"Start time: {time.strftime('%H:%M:%S', time.localtime(start_time))}")

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM cases WHERE opinions_vector IS NULL")
    total_rows = cur.fetchone()[0]
    print(f"Total rows to process: {total_rows}")

    cur.execute("SELECT id, name, opinion FROM cases WHERE opinions_vector IS NULL ORDER BY id")
    rows = cur.fetchall()

processed = 0
for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i : i + BATCH_SIZE]
    texts = [(name or "") + " " + (opinion or "")[:8000] for _id, name, opinion in batch]

    # Call Azure OpenAI embeddings
    response = embed_client.embeddings.create(
        input=texts,
        model=AZURE_EMBED_DEPLOYMENT,
    )

    # Update each row with its embedding
    with conn.cursor() as cur:
        for j, item in enumerate(response.data):
            vector_str = "[" + ",".join(str(v) for v in item.embedding) + "]"
            cur.execute(
                "UPDATE cases SET opinions_vector = %s::vector WHERE id = %s",
                (vector_str, batch[j][0]),
            )

    processed += len(batch)
    pct = round(100.0 * processed / total_rows, 1)
    print(f"Progress: {processed} / {total_rows} ({pct}%)")

conn.close()

end_time = time.time()
duration = end_time - start_time
print(f"\nEnd time: {time.strftime('%H:%M:%S', time.localtime(end_time))}")
print(f"Total duration: {int(duration // 60)}m {duration % 60:.1f}s")
print(f"Done. {processed} rows processed.")

### Part 2.10: Build the case citation graph with Apache AGE

##### 🧠 *Technical Background Notes*

Apache AGE turns Postgres into a **multi-model** store - relational and property-graph in the same database, in the same transaction. Here we model every court case as a node (label `case`) and every citation as a directed edge (relationship type `REF`), so the agent can later reason about which precedents support which opinions - all queryable in one connection / one transaction alongside the relational rows and vector embeddings.

Each `:case` node carries the columns the agent will most often need at hand:

- `case_id` - the canonical id, also used to match edge endpoints
- `name` - short case name, e.g. *"Smith v. Jones"*
- `decision_date` - date as ISO text
- `court_level` - e.g. *"Supreme Court of Pennsylvania"*
- `opinion` - concatenated opinion text

Storing them on the node means a single Cypher traversal can return the citing context without a second SQL round-trip.

Highlights of the cell:

- **Two PL/pgSQL helpers** - `create_case_in_case_graph` and `create_case_link_in_case_graph` wrap AGE's `cypher(...)` function. They use the 3-argument form `cypher(graph, query, agtype_build_map(...))` so values are passed as **bound parameters** (`$name`, `$case_id`, ...) - no manual string escaping, safe for opinion text full of quotes and newlines.
- **Idempotent rebuild** - the `DO $$ ... drop_graph ... $$` block drops `case_graph` only if it exists, then `create_graph('case_graph')` re-creates it. Safe to re-run.
- **Nodes from `cases`** - we pull node properties from the typed `cases` table (already parsed out of the raw JSON in Part 2.5). `id::text` here matches the string `case_id` the edge step looks up below.
- **Edges from JSON arrays** - the final CTE uses `LATERAL jsonb_array_elements` on `temp_cases.data` to explode `cites_to[*].case_ids[*]`, then inner-joins `temp_cases` back to itself to keep only edges whose target is in our dataset.

`autocommit=True` is required because `CREATE EXTENSION` cannot run inside a transaction block and AGE's `drop_graph` / `create_graph` perform their own DDL internally. `SET search_path = public, ag_catalog, "$user"` ensures unqualified calls like `create_graph(...)` resolve to AGE's functions.

##### 📝 *Tasks*



1. Run the cell below using the "▶" icon next to the cell.    > **Note:** Edge creation is the slowest step in this cell - expect several minutes.



1. After `Graph nodes created.`, confirm the `Node count:` line matches the number of cases loaded earlier.1. Edge creation prints incremental `edges: X/Y (Z%) - N edges/s` progress lines, finishing with `Done. Case graph is ready.`


In [ ]:
# Build the case citation graph using Apache AGE
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    # ------------------------------------------------------------------------
    # 0) Make sure ag_catalog is on the search path *now*, before we CREATE
    #    the helper functions below. PL/pgSQL resolves the `agtype` type at
    #    function-create time, so without this we'd get:
    #        UndefinedObject: type "agtype" does not exist
    #    (AGE itself was already installed in Part 2.4.)
    # ------------------------------------------------------------------------
    cur.execute("CREATE EXTENSION IF NOT EXISTS age CASCADE;")
    cur.execute("SET search_path = public, ag_catalog, \"$user\";")

    # ------------------------------------------------------------------------
    # 1) Helper functions: thin SQL wrappers around AGE's `cypher(...)` call.
    #
    # AGE exposes Cypher through cypher(graph_name, query_text, params_agtype).
    # The third (params) argument is a quirk: AGE *parses* the call and only
    # accepts it if the third argument is a literal parameter reference
    # (e.g. `$1`). Passing `agtype_build_map(...)` inline raises:
    #
    #     "third argument of cypher function must be a parameter"
    #
    # The workaround used below: build the agtype map into a PL/pgSQL
    # variable, then run the cypher() call through `EXECUTE ... USING props`
    # so PL/pgSQL substitutes `$1` as a real bind parameter. This is also
    # the safest way to ship arbitrary string values (quotes, newlines,
    # unicode) into Cypher without any manual escaping.
    # ------------------------------------------------------------------------

    # 1a) Create a single :case node with all of its properties.
    cur.execute("""
        CREATE OR REPLACE FUNCTION create_case_in_case_graph(
            case_id       text,
            name          text,
            decision_date text,
            court_level   text,
            opinion       text
        )
        RETURNS void
        LANGUAGE plpgsql
        VOLATILE
        AS $BODY$
        DECLARE
            props ag_catalog.agtype;
        BEGIN
            -- AGE's cypher() and agtype_build_map() live in ag_catalog; put
            -- them on the search path for the duration of this function call.
            SET search_path TO ag_catalog;

            -- Build the parameter map. Each key here is referenced as
            -- $key inside the Cypher text below.
            props := agtype_build_map(
                'case_id',       case_id,
                'name',          name,
                'decision_date', decision_date,
                'court_level',   court_level,
                'opinion',       opinion
            );

            -- EXECUTE ... USING binds props as $1, which is what AGE
            -- requires for the third cypher() argument.
            EXECUTE $sql$
                SELECT * FROM cypher(
                    'case_graph',
                    $cy$
                        CREATE (:case {
                            case_id:       $case_id,
                            name:          $name,
                            decision_date: $decision_date,
                            court_level:   $court_level,
                            opinion:       $opinion
                        })
                    $cy$,
                    $1
                ) AS (a agtype);
            $sql$ USING props;
        END
        $BODY$;
    """)

    # 1b) Create a directed :REF edge from one case node to another.
    #     Same EXECUTE ... USING pattern as 1a.
    cur.execute("""
        CREATE OR REPLACE FUNCTION create_case_link_in_case_graph(id_from text, id_to text)
        RETURNS void
        LANGUAGE plpgsql
        VOLATILE
        AS $BODY$
        DECLARE
            props ag_catalog.agtype;
        BEGIN
            SET search_path TO ag_catalog;

            props := agtype_build_map('id_from', id_from, 'id_to', id_to);

            EXECUTE $sql$
                SELECT * FROM cypher(
                    'case_graph',
                    $cy$
                        MATCH (a:case), (b:case)
                        WHERE a.case_id = $id_from AND b.case_id = $id_to
                        CREATE (a)-[e:REF]->(b)
                        RETURN e
                    $cy$,
                    $1
                ) AS (a agtype);
            $sql$ USING props;
        END
        $BODY$;
    """)
    print("Helper functions created.")

    # ------------------------------------------------------------------------
    # 2) Drop any previous case_graph so this cell is safely re-runnable.
    #
    #    drop_graph() raises an error if the graph doesn't exist, so we guard
    #    it with a check against ag_catalog.ag_graph (AGE's metadata table).
    #    The second arg `true` = cascade (also drop the graph's backing tables).
    # ------------------------------------------------------------------------
    cur.execute("""
        DO $$
        BEGIN
            IF EXISTS (SELECT 1 FROM ag_catalog.ag_graph WHERE name = 'case_graph') THEN
                PERFORM ag_catalog.drop_graph('case_graph', true);
            END IF;
        END $$;
    """)
    print("Old case_graph dropped (if it existed).")

    # ------------------------------------------------------------------------
    # 3) Create a fresh, empty graph named 'case_graph'.
    # ------------------------------------------------------------------------
    cur.execute("SELECT create_graph('case_graph');")
    print("case_graph created.")

    # ------------------------------------------------------------------------
    # 4) Populate the nodes.
    #
    #    Pull node properties from the typed `cases` table (already parsed
    #    out of the raw JSON in Part 2.5). `id::text` matches the case_id
    #    string the citation edges will reference below.
    # ------------------------------------------------------------------------
    cur.execute("""
        SELECT create_case_in_case_graph(
            id::text,
            name,
            decision_date::text,
            court_level,
            opinion
        )
        FROM public.cases;
    """)
    print("Graph nodes created.")

    # ------------------------------------------------------------------------
    # 5) Sanity check: count the nodes back out via Cypher.
    #    This proves both the graph and the cypher() function are healthy
    #    before we spend time creating edges.
    # ------------------------------------------------------------------------
    cur.execute("""
        SELECT * FROM cypher('case_graph', $$
            MATCH (n)
            RETURN COUNT(n.case_id)
        $$) AS (case_id TEXT);
    """)
    node_count = cur.fetchone()[0]
    print(f"Node count: {node_count}")

    # ------------------------------------------------------------------------
    # 6) Build the citation edges.
    #
    #    Each row in temp_cases has a JSON array `cites_to`, and each entry
    #    in that array has its own `case_ids` array of cited cases:
    #
    #        data.cites_to = [
    #            { "case_ids": ["123", "456"], ... },
    #            { "case_ids": ["789"], ... },
    #        ]
    #
    #    LATERAL jsonb_array_elements flattens both levels:
    #       c1 -> cites_to_element -> case_ids   (one row per cited id)
    #
    #    We then INNER JOIN back to temp_cases (c2) on the cited id, which
    #    filters out citations pointing to cases outside our dataset - AGE
    #    would happily create dangling edges otherwise.
    #
    #    Edge creation is the slowest step (one cypher() call per edge), so
    #    we materialise the (from, to) pairs first, then loop in Python and
    #    print a progress line every PROGRESS_EVERY edges.
    # ------------------------------------------------------------------------
    cur.execute("""
        WITH edges AS (
            SELECT c1.data#>>'{id}' AS id_from, c2.data#>>'{id}' AS id_to
            FROM public.temp_cases c1
            LEFT JOIN
                LATERAL jsonb_array_elements(c1.data -> 'cites_to') AS cites_to_element ON true
            LEFT JOIN
                LATERAL jsonb_array_elements(cites_to_element -> 'case_ids') AS case_ids ON true
            JOIN public.temp_cases c2
                ON case_ids::text = c2.data#>>'{id}'
        )
        SELECT id_from::text, id_to::text FROM edges;
    """)
    # cur.fetchall() pulls the entire (id_from, id_to) result set into memory
    # as a Python list of tuples. The edge count is in the low thousands here,
    # so this is cheap - for a much larger graph you'd switch to a server-side
    # named cursor and stream rows instead.
    edge_pairs = cur.fetchall()
    total_edges = len(edge_pairs)
    print(f"Creating {total_edges} citation edges...")

    # PROGRESS_EVERY controls how often we print a progress line. We aim for
    # roughly 20 updates across the whole loop (one every ~5%). max(1, ...)
    # guards against the degenerate case where total_edges < 20, which would
    # otherwise make the divisor 0 and trigger a ZeroDivisionError on `i %`.
    PROGRESS_EVERY = max(1, total_edges // 20)

    # Capture a wall-clock start time so we can report an instantaneous
    # edges-per-second throughput alongside the percent-complete number.
    edge_start = time.time()

    # Loop over the materialised pairs and create one :REF edge per pair by
    # calling the PL/pgSQL helper defined in step 1b. enumerate(..., start=1)
    # gives us a 1-based counter so the progress output reads naturally
    # ("1/1234" rather than "0/1234").
    for i, (id_from, id_to) in enumerate(edge_pairs, start=1):
        cur.execute(
            "SELECT public.create_case_link_in_case_graph(%s, %s);",
            (id_from, id_to),
        )

        # Print progress on every Nth edge, and also on the very last edge
        # so the final "100.0%" line is guaranteed to appear even when
        # total_edges isn't a multiple of PROGRESS_EVERY.
        if i % PROGRESS_EVERY == 0 or i == total_edges:
            elapsed = time.time() - edge_start
            # Guard against divide-by-zero on the (very unlikely) case where
            # the first batch completes in under the timer's resolution.
            rate = i / elapsed if elapsed > 0 else 0
            pct = 100.0 * i / total_edges
            print(f"  edges: {i}/{total_edges} ({pct:.1f}%) - {rate:.1f} edges/s")

    print("Citation edges created.")

conn.close()
print("Done. Case graph is ready.")


### Part 2.11: Build the full-text search index with BM25 search

##### 🧠 *Technical Background Notes*

`pg_fts` is HorizonDB's next-generation full-text engine - a drop-in replacement for `tsvector`/`tsquery` with better ranking, phrase search, and no need to maintain a separate `tsvector` column.

- **Multi-column index** - `USING fts (name text_fts_ops, opinion text_fts_ops)` builds one index covering both fields, so a single query can match either.
- **Query operator** - `fts_query('water leaking', 'idx_cases_fts')` is a boolean predicate that returns `true` when a row matches; the planner uses the named index directly.
- **`hello_pg_fts()`** is just a smoke test that the extension loaded and the `pgfts` schema is on the search path.

Later in the lab the agent will combine this lexical index with vector similarity to do **hybrid search** - exact keywords from `pg_fts` plus semantic recall from `pg_diskann`.


##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. The output should show `pg_fts extension is ready.` followed by a `Matching cases:` count for the sample `water leaking` query.

In [ ]:
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    # Drop existing tables
    cur.execute("CREATE EXTENSION IF NOT EXISTS pg_fts;")
    cur.execute("SET search_path = public, pgfts;")
    cur.execute("SELECT hello_pg_fts();")

    result = cur.fetchone()

    print(f"pg_fts extension is ready. Result: {result}")
   
    cur.execute("DROP INDEX IF EXISTS idx_cases_fts;")
    cur.execute("""
        CREATE INDEX idx_cases_fts
        ON public.cases
        USING fts (name text_fts_ops, opinion text_fts_ops);
        """)
    
    cur.execute("""
        SELECT count(*) FROM cases
        WHERE fts_query('water leaking', 'idx_cases_fts')
        """)
    
    count = cur.fetchone()[0]

    print(f"fts_query executed. Matching cases: {count}")

conn.close()

### Part 2.12: Build the DiskANN vector index

##### 🧠 *Technical Background Notes*

`pg_diskann` is Microsoft's implementation of the DiskANN ANN algorithm to Postgres. It uses a hybrid in-memory and on-disk architecture, which is what lets HorizonDB serve millions of vectors per node without blowing memory.

- **`vector_cosine_ops`** - cosine distance matches how Azure OpenAI embeddings are normalised. Use `vector_l2_ops` for raw L2 distance or `vector_ip_ops` for inner product.
- **`product_quantized = 'false'`** - PQ compression isn't yet available on HorizonDB, so we build a full-precision index. When PQ ships you'll be able to trade a bit of recall for a much smaller index.
- **`IF NOT EXISTS`** - re-running the cell is a no-op once the index exists; rebuilding requires a `DROP INDEX` first.
- **Context-manager connect** - `with psycopg.connect(..., autocommit=True) as conn, conn.cursor() as cur:` is the modern, leak-free pattern; both `cur` and `conn` close automatically on exit.

With this index in place, every `ORDER BY opinions_vector <=> :query_vec LIMIT k` query the agent runs becomes an index scan instead of a sequential one.


##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. A single `pg_diskann ready: idx_cases_diskann on public.cases(opinions_vector)` line confirms the index was created.

    > **Note:** Building the DiskANN index over all opinion vectors can take a couple of minutes.

In [ ]:
with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("CREATE EXTENSION IF NOT EXISTS pg_diskann;")

    # DiskANN index (quantization not supported yet on HorizonDB)
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_cases_diskann
        ON public.cases
        USING diskann (opinions_vector vector_cosine_ops)
        WITH (product_quantized='false');
    """)
print("pg_diskann ready: idx_cases_diskann on public.cases(opinions_vector)")